# Team CopyPaste
## Finetuning code for our model
## Instructions: Run each cell one by one

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2

In [2]:
import os
import torch
import pandas as pd
from datasets import Dataset

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-06 12:22:30.120521: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765023750.558160      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765023750.678768      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
import torch
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
model_name = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"

max_seq_length = 16384  # or 4096 / 8192 depending on your GPU
dtype = torch.bfloat16

model, tokenizer = FastVisionModel.from_pretrained(
    model_name          = model_name,
    max_seq_length      = max_seq_length,
    load_in_4bit        = True,
    torch_dtype         = dtype,
    use_gradient_checkpointing = "unsloth",
)


==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [4]:
model = FastVisionModel.get_peft_model(
    model,
    # In the video they leave vision layers frozen
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r           = 16,
    lora_alpha  = 16,
    lora_dropout= 0.0,
    bias        = "none",
    random_state= 3407,
    use_gradient_checkpointing = "unsloth"
)

try:
  model.print_trainable_parameters()
except Exception as e:
  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total = sum(p.numel() for p in model.parameters())
  print(f"Trainable: {trainable} | Total: {total} | {trainable/total*100:.2f}%")


trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954


## Add dataset from `Input`

In [5]:
from datasets import Dataset
from PIL import Image
import pandas as pd
import os

# Paths from your description
CSV_PATH = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Train.csv"
IMG_DIR = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Image"

# 1. Load first 500 rows
train_df = pd.read_csv(CSV_PATH).head(2800)

TEXT_CONTENT = """You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

# 2. Create a HF Dataset with raw columns first
#    We keep Image_name and Label, we'll add decoded_image
raw_ds = Dataset.from_pandas(train_df, preserve_index=False)

# 3. Load, resize to 512x512, convert to RGB -> decoded_image
def load_and_resize(example):
    img_path = os.path.join(IMG_DIR, example["Image_name"])
    img = Image.open(img_path)
    img = img.resize((512, 512))
    if img.mode != "RGB":
        img = img.convert("RGB")
    example["decoded_image"] = img
    return example

dataset = raw_ds.map(load_and_resize)

# 4. Build MathVista-style prompt structure: list with image placeholder + text
def make_conversation(example):
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # placeholder; actual image kept in decoded_image
                {"type": "text", "text": TEXT_CONTENT},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["Label"]},
            ],
        },
    ]
    return {
        "prompt": prompt,
        "image": example["decoded_image"],  # keep image separate (like MathVista)
        "answer": example["Label"],
    }

dataset = dataset.map(make_conversation)

# 5. Remove old image column (if any) and rename decoded_image -> image
#    In our case, we already use 'image' key above, so we can just drop decoded_image.
if "decoded_image" in dataset.column_names:
    dataset = dataset.remove_columns("decoded_image")

# 6. Apply chat template to build final text prompt string
#    tokenizer must already be created (from your FastVisionModel)
def apply_template(example):
    example["prompt"] = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=False,
        add_generation_prompt=True,  # include assistant tag
    )
    return example

dataset = dataset.map(apply_template)

print(dataset)
print("Columns:", dataset.column_names)
print("Sample prompt:", dataset[0]["prompt"][:400])
print("Image type:", type(dataset[0]["image"]))


Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Dataset({
    features: ['Image_name', 'Label', 'prompt', 'image', 'answer'],
    num_rows: 2800
})
Columns: ['Image_name', 'Label', 'prompt', 'image', 'answer']
Sample prompt: <|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non pol
Image type: <class 'PIL.PngImagePlugin.PngImageFile'>


## Reinforcement Learning Reward function

In [7]:
import re
import math

def meme_correctness_reward_func_balanced(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Domain-Aware Reward Function (DISCO approach) for imbalanced classification.

    For the dataset (853 political, 2007 non-political):
    - Political weight: 1.47 (minority class gets higher weighting)
    - Non-political weight: 0.88 (majority class gets lower weighting)

    Reward Scaling:
    - Correct political: +2.94 (2.0 * 1.47)
    - Wrong political: -2.94 (penalize harder to encourage learning)
    - Correct non-political: +1.76 (2.0 * 0.88)
    - Wrong non-political: -1.76
    
    """

    # Domain frequencies 
    political_freq = 0.298  # 853/2860
    non_political_freq = 0.702  # 2007/2860

    # Calculate domain-aware weights using log scaling (DISCO formula)
    political_weight = math.log(1 + 1/political_freq)      # log(4.36) ≈ 1.47
    non_political_weight = math.log(1 + 1/non_political_freq)  # log(2.42) ≈ 0.88

    # Debug logging
    q = prompts[0]
    print(
        "-" * 20,
        f"Question:\n{q}",
        f"\nAnswer:\n{answer[0]}",
        f"\nResponse:{completions[0]}",
    )

    def get_label(text: str) -> str:
        """Extract predicted label from model completion"""
        text = text.strip()
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        text = re.sub(r"<.*?>", "", text).strip()
        pred = text.lower()

        if "nonpolitical" in pred or "non-political" in pred or "non political" in pred:
            return "nonpolitical"
        if "political" in pred:
            return "political"
        return ""

    def gold_label(a: str) -> str:
        """Extract ground truth label"""
        g = a.lower().strip()
        return "nonpolitical" if "non" in g else "political"

    def get_class_weight(label: str) -> float:
        """Get domain-aware weight for class"""
        return political_weight if label == "political" else non_political_weight

    # Compute rewards with domain-aware scaling
    rewards = []
    for completion, ground_truth in zip(completions, answer):
        predicted = get_label(completion)
        true_label = gold_label(ground_truth)
        is_correct = (predicted == true_label)
        class_weight = get_class_weight(true_label)

        # Apply domain-aware scaling
        base_reward = 2.0
        reward = (base_reward * class_weight) if is_correct else (-base_reward * class_weight)
        rewards.append(reward)

    return rewards


## Testing for correct format and output

In [8]:
dataset[100]["prompt"]

'<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:\nPolitical: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.\nNonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.\nOutput a single word without providing any reasoning or explanations<|im_end|>\n<|im_start|>assistant\nNonPolitical<|im_end|>\n<|im_start|>assistant\n'

In [9]:
image = dataset[100]["image"]
prompt = dataset[100]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

NonPolitical<|im_end|>


## Training config and trainer

In [10]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    log_completions = False,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 0.5, # Set to 1 for a full training run
    # max_steps = 60,
    save_steps = 60,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

In [11]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
      meme_correctness_reward_func_balanced
    ],
    train_dataset = dataset,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,800 | Num Epochs = 1 | Total steps = 700
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 43,646,976 of 8,810,770,672 (0.50% trained)
`generation_config` default values have been modified to match model-specific defaults: {'temperature': 0.7, 'top_p': 0.8}. If this is not desired, please set these values explicitly.


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
Political<|im_end|>
<|im_start|>assistant
 
Answer:
Political 
Response:Political
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / meme_correctness_reward_func_balanced / mean,rewards / meme_correctness_reward_func_balanced / std
1,0.000000,2.942973,0.000000,2.000000,2.000000,2.000000,0.000000,2.000000,2.000000,2.000000,0.000000,2.942973,0.000000
2,0.000000,2.357112,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.357112,0.676493
3,0.000000,0.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000002,0.000000,3.398252
4,-0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,-0.000000,1.771252,0.000000
5,0.000000,2.357112,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.357112,0.676493
6,0.000000,2.357112,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.357112,0.676493
7,-0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,-0.000000,1.771252,0.000000
8,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,1.771252,0.000000
9,0.000000,0.000000,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000010,0.000000,3.398252
10,-0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,-0.000000,1.771252,0.000000


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
NonPolitical<|im_end|>
<|im_start|>assistant
 
Answer:
NonPolitical 
Response:NonPolitical
-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a p


KeyboardInterrupt



# If training freezees at any point, we can resume from a checkpoint

In [ ]:
trainer.train(resume_from_checkpoint="outputs/checkpoint-60")


The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,800 | Num Epochs = 1 | Total steps = 700
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 43,646,976 of 8,810,770,672 (0.50% trained)


-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
NonPolitical<|im_end|>
<|im_start|>assistant
 
Answer:
NonPolitical 
Response:NonPolitical


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / meme_correctness_reward_func_balanced / mean,rewards / meme_correctness_reward_func_balanced / std
61,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,1.771252,0.000000
62,0.000200,1.471486,2.080996,2.250000,2.000000,3.000000,0.000000,2.250000,2.000000,3.000000,0.001715,1.471486,2.942973
63,0.000200,0.885626,2.080996,2.750000,2.000000,3.000000,0.000000,2.750000,2.000000,3.000000,0.002452,0.885626,2.611482
64,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,1.771252,0.000000
65,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,1.771252,0.000000
66,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,1.771252,0.000000
67,0.000000,2.942973,0.000000,2.000000,2.000000,2.000000,0.000000,2.000000,2.000000,2.000000,0.000000,2.942973,0.000000
68,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,1.771252,0.000000
69,0.000000,1.771252,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000407,1.771252,0.000000
70,0.000000,2.357112,0.000000,2.500000,2.000000,3.000000,0.000000,2.500000,2.000000,3.000000,0.000002,2.357112,0.676493


Streaming output truncated to the last 5000 lines.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
NonPolitical<|im_end|>
<|im_start|>assistant
 
Answer:
NonPolitical 
Response:NonPolitical
-------------------- Question:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations<|im_end|>
<|im_start|>assistant
NonPolitical<|im_end|>
<|im_start|>assistant
 
Answer:
NonPolitical 
Response:NonPolitical
-------------

TrainOutput(global_step=700, training_loss=1.0263941338236343e-06, metrics={'train_runtime': 6345.9051, 'train_samples_per_second': 0.221, 'train_steps_per_second': 0.11, 'total_flos': 0.0, 'train_loss': 1.0263941338236343e-06})

## Trained model is saved to hugging for inference later on

In [ ]:
model.push_to_hub_merged("arafid/Q3-8B-GRPO-Balanced-Vision-Freezed", tokenizer, save_method = "merged_16bit", token = "<Hugging Face Write Token>")

config.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...on-Freezed/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   1%|1         | 33.5MB / 2.72GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [12:35<00:00, 188.86s/it]


Unsloth: Merge process complete. Saved to `/content/arafid/Q3-8B-GRPO-Balanced-Vision-Freezed`
